# SGD versus momentum: interference and neuron movements

Teacher 8, students 8/16, both layers train. Main settings: SGD lr 1;
momentum lr 0.1, beta 0.9, no Nesterov. An SGD-lr-0.1 pilot supplies a later control.
All analysis uses existing states; no training is run by this notebook.

**Main difference:** greater width improves the early momentum direction even
when raw P, N and A barely change. Partial cancellation increases at greater width
for both optimizers; R remains above one. Momentum's early reduction in normalized
harm persists for 1% steps. These benefits are not uniform across training phases.

See [the report](../docs/momentum_comparison.md) for definitions and limitations.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT=Path.cwd()
if not (ROOT/'pyproject.toml').exists():
    ROOT=ROOT.parent
OUT=ROOT/'runs/pier_study/momentum'
grad=pd.read_csv(OUT/'gradient_summary.csv')
actual=pd.read_csv(OUT/'actual_step_summary.csv')
movement=pd.read_csv(OUT/'movement_summary.csv')
effects=pd.read_csv(OUT/'movement_effects.csv')
contrasts=pd.read_csv(OUT/'optimizer_width_contrasts.csv')


## 1. Raw gradients and actual directions

Raw $q_r=g_{B,r}^{\top}g_{C,r}$ gives P/N before summing neurons; A is the
full-gradient squared norm divided by mean minibatch gradient energy.

For momentum, candidate directions are
$h_B=\beta v_{t-1}+g_B$, using the **pre-step** buffer. Positive
$g_C^{\top}h_B$ predicts help to C. Direction-transfer masses use normalization
$\sqrt{\mathbb E\|g_C\|^2\mathbb E\|h_B\|^2}$ and average ordered off-diagonal
pairs because the transfer is asymmetric. Holding the buffer fixed does not
create independent optimizer histories; it examines possible next steps from one history.


In [ ]:
early=grad[(grad.comparison=='mse_0.2')&(grad.probes==512)]
display(early[['optimizer','width','n_seeds','train_mse_mean','P','N','A',
               'p_negative','step_P','step_N','step_p_negative','skewness','excess_kurtosis']].round(6))
display(actual[(actual.comparison=='mse_0.2')&(actual.probes==512)]
        [['optimizer','width','scale','linear_harm_fraction','finite_harm_fraction']].round(6))
display(Image(filename=str(OUT/'momentum_gradients.png')))


## 2. Agreement and geometry over training

Ten seeds per profile at shared checkpoints. These are equal-step comparisons,
which need not have equal losses. Late ratios for well-fitted wide students may
involve numerical floors. Higher raw A is not a general predictor of lower loss
and does not measure momentum's actual direction.


In [ ]:
display(Image(filename=str(OUT/'momentum_agreement.png')))
trajectory=pd.read_csv(OUT/'trajectory_summary.csv')
display(trajectory[trajectory.comparison=='mse_0.2']
        [['optimizer','width','batch_sharpness_ratio','step_previous_cosine','step_norm']].round(6))


## 3. Exact neuron movements

Each $d_r$ is the exact finite prediction change from updating neuron r.
Partial cancellation is $1-\mathbb E|\sum d_r|/\mathbb E\sum|d_r|$;
$R=\mathbb E(\sum d_r)^2/\mathbb E\sum d_r^2$ distinguishes net cancellation
(R<1) from net reinforcement of prediction movement (R>1).

Accurate examples satisfy $|f-y|\le .05$ at BOTH widths within an optimizer/seed.
This subset need not be the same across optimizers. Candidate updates use 32 source
batches; loss effects are measured on 4,096 independent test examples.


In [ ]:
chosen=movement[(movement.comparison_kind=='matched_loss')&(movement.direction=='optimizer')&
                 (movement.scale==1)&(movement.subset=='well_fit_0.05')]
display(chosen[['optimizer','comparison','width','n_seeds','n_examples_mean','R',
                'cancellation_fraction','solo_harm_offset_fraction',
                'joint_harm_per_source_gain','joint_harm_per_source_gain_n_valid']].round(6))
display(Image(filename=str(OUT/'momentum_neurons.png')))


## 4. Step-length sensitivity and benefits as well as harm

H/U divides positive pointwise held-out loss changes by mean source-batch
improvement; G/U divides beneficial changes by the same U. Net transfer is
(G-H)/U. All sampled states have positive mean U, although individual momentum
steps can harm their source batch. Those steps are retained.

Shortening to 1% changes a local diagnostic step, not the preceding training.
The gradient-only control removes the buffer from candidate steps at the SAME
momentum-trained state, using the same nominal learning rate.


In [ ]:
early_move=movement[(movement.comparison=='mse_0.2')&movement.subset.isin(['all','well_fit_0.05'])]
display(early_move[['optimizer','direction','scale','subset','width','joint_harm_per_source_gain',
                    'joint_benefit_per_source_gain','net_transfer_per_source_gain',
                    'source_harm_fraction']].round(6))
display(Image(filename=str(OUT/'momentum_step_sensitivity.png')))


## 5. Fresh-batch checks and uncertainty

Fresh source batches use seed 20261018. Early contrasts use all 40 common seeds;
later cross-optimizer contrasts have six common seeds, distinct from the eight
SGD and seven momentum width-paired samples. A contrast below is the momentum
width effect minus the SGD width effect. Intervals are exploratory paired
seed-bootstrap intervals conditional on the fixed teacher and data.


In [ ]:
fresh=pd.read_csv(OUT/'fresh_movement_summary.csv')
display(fresh[(fresh.optimizer=='momentum_01')&(fresh.direction=='optimizer')&
               (fresh.subset=='well_fit_0.05')]
        [['scale','width','cancellation_fraction','R','joint_harm_per_source_gain']].round(6))
display(contrasts[(contrasts.domain=='movement')&(contrasts.comparison=='mse_0.2')&
                  (contrasts.subset=='well_fit_0.05')&
                   contrasts.metric.isin(['cancellation_fraction','joint_harm_per_source_gain'])]
        [['scale','metric','n','mean','low','high']].round(6))
display(json.loads((OUT/'audit.json').read_text()))


The early direction-transfer result is clearer with momentum, but it does
not establish that neuron cancellation causes the final width advantage.
R stays above one, the later harm benefit is uncertain, and both optimizers have
similar width-8/16 final fitting success in these settings. Raw gradient geometry,
optimizer history, finite-step effects and useful net progress are separate
measurements.
